# Exportar dados do Lakehouse para o Google Sheets

```
Lakehouse (Tables/sales_summary) ──► Notebook ──► Google Sheets (aba sales_summary)
                                        ▲
              Azure Key Vault ──────────┘  JSON da service account do Google
```

- **Autenticação:** a chave da service account fica no Azure Key Vault e é lida em tempo de execução. Nada de segredo no código.
- **Carga:** full. A cada execução a aba é **limpa e reescrita**: mesma planilha, mesma aba, sem acumular linhas.

In [ ]:
%pip install gspread google-auth -q

## Parâmetros

Célula marcada como *parameter cell*: pode ser sobrescrita por um pipeline ou por `notebookutils.notebook.run`.

In [ ]:
KEY_VAULT_URI = "https://<key-vault-name>.vault.azure.net/"
SECRET_NAME = "gcp-sa-sheets-json"
SPREADSHEET_KEY = "<spreadsheet-key>"  # the part of the URL between /d/ and /edit
WORKSHEET_NAME = "sales_summary"
SOURCE_SCHEMA = "dbo"
SOURCE_TABLE = "sales_summary"

## 1. Autenticar no Google com a service account guardada no Key Vault

In [ ]:
import json
import math
from datetime import date, datetime, timezone
from decimal import Decimal
from typing import Any

import gspread
import notebookutils  # provided by the Fabric runtime
import pandas as pd
from deltalake import DeltaTable
from google.oauth2.service_account import Credentials

SCOPES = ["https://www.googleapis.com/auth/spreadsheets"]

# getSecret runs as the identity executing the notebook: it needs "Key Vault Secrets User".
service_account_info: dict[str, Any] = json.loads(
    notebookutils.credentials.getSecret(KEY_VAULT_URI, SECRET_NAME)
)
credentials = Credentials.from_service_account_info(service_account_info, scopes=SCOPES)
gc = gspread.authorize(credentials)

# Never print the secret itself; the account e-mail is enough to confirm it loaded.
print(f"Authenticated as {credentials.service_account_email}")

## 2. Ler a tabela Delta do lakehouse

In [ ]:
df = (
    # Reading through the FUSE mount is fine; only writes need the abfss endpoint.
    DeltaTable(f"/lakehouse/default/Tables/{SOURCE_SCHEMA}/{SOURCE_TABLE}")
    .to_pandas()
    .sort_values(["order_month", "region", "product_category"])
    .reset_index(drop=True)
)
print(f"Rows read from {SOURCE_SCHEMA}.{SOURCE_TABLE}: {len(df)}")
df.head()

## 3. Converter para o formato aceito pela API do Sheets

A API recebe JSON: datas viram texto ISO, `Decimal` vira `float` e nulos viram célula vazia.

In [ ]:
SheetCell = str | int | float
SHEETS_CELL_LIMIT = 10_000_000  # Google Sheets limit per spreadsheet


def to_cell(value: Any) -> SheetCell:
    """Convert one pandas/Arrow value into a JSON-serializable Sheets cell."""
    if value is None or value is pd.NA or value is pd.NaT:
        return ""
    if isinstance(value, float) and math.isnan(value):
        return ""
    if isinstance(value, Decimal):
        return float(value)
    if isinstance(value, (datetime, date)):
        return value.isoformat()
    if hasattr(value, "item"):  # numpy scalar
        return to_cell(value.item())
    if isinstance(value, (bool, int, float, str)):
        return value
    return str(value)


def to_sheet_values(frame: pd.DataFrame) -> list[list[SheetCell]]:
    """Return header + rows, ready for Worksheet.update."""
    header: list[SheetCell] = [str(column) for column in frame.columns]
    rows = [[to_cell(value) for value in row] for row in frame.itertuples(index=False)]
    return [header, *rows]


values = to_sheet_values(df)
row_count, col_count = len(values), len(values[0])
assert row_count * col_count < SHEETS_CELL_LIMIT, "Too many cells for Google Sheets"
print(f"Prepared {row_count - 1} rows x {col_count} columns")

## 4. Escrever na planilha (sobrescreve a aba)

A planilha já existe e foi compartilhada com a service account. A aba é criada só na primeira execução.

In [ ]:
spreadsheet = gc.open_by_key(SPREADSHEET_KEY)

try:
    worksheet = spreadsheet.worksheet(WORKSHEET_NAME)
except gspread.WorksheetNotFound:
    worksheet = spreadsheet.add_worksheet(title=WORKSHEET_NAME, rows=row_count, cols=col_count)

worksheet.clear()
worksheet.resize(rows=row_count, cols=col_count)
worksheet.update(range_name="A1", values=values, value_input_option="USER_ENTERED")
worksheet.freeze(rows=1)

## 5. Resumo

In [ ]:
exported_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
print(f"Exported {row_count - 1} rows to '{WORKSHEET_NAME}' at {exported_at}")
print(f"https://docs.google.com/spreadsheets/d/{SPREADSHEET_KEY}")